<h1> <a id="title"></a>Differencing and geostatistical error analysis in lidar topographic differencing: Workflow starting with user provided point clouds</h1>

This work was funded by the [John Wesley Powell Center for Analysis and Synthesis (USGS G23AC00336)](https://www.usgs.gov/centers/john-wesley-powell-center-for-analysis-and-synthesis/science/a-national-topographic-change#overview) and [OpenTopography](https://opentopography.org).  OpenTopography is supported by the National Science Foundation under Awards # 2410799, 2410800 & 2410801.

<h2><a id="introduction"></a>1. Introduction</h2>

Vertical topographic differencing calculates net change in the vertical dimension by comparing digital elevation models (DEMs) collected at different times ([Izumida et al., 2017](https://doi.org/10.5194/nhess-17-1505-2017); [Wheaton et al., 2010](https://doi.org/10.1002/esp.1886)). Topographic differencing underpins a wide range of studies, spanning vegetation biomass changes, lava-flow emplacement ([Albino et al., 2015](https://doi.org/10.1002/2015JB011988)), fluvial and coastal floods ([Izumida et al., 2017](https://doi.org/10.5194/nhess-17-1505-2017)), landslides ([Lucieer et al., 2014](https://doi.org/10.1177/0309133313515293)), fine-scale fluvial sediment budgets ([Wheaton et al., 2010](https://doi.org/10.1002/esp.1886)), and tectonic activity ([Langridge et al., 2014](https://doi.org/10.1016/j.geomorph.2014.08.007); [Scott et al., 2018](https://doi.org/10.1029/2018JB015581)).

<h3><a id="differencing_def"></a>Vertical topographic differencing</h3>

Vertical differencing is the pixel-by-pixel subtraction of two digital-elevation models (DEMs) that share a common coordinate system and identical grid geometry (e.g., [Scott et al., 2021](https://doi.org/10.1130/GES02259.1)). Because every cell occupies the same planimetric position in both rasters, subtracting them yields a raster of elevation change ($\Delta z$). Researchers rarely inspect the differencing value of every cell individually. Instead, aggregate change is summarized over a specific feature or landform (net sediment deposition on a bar, net tree growth in a reforested stand, net inflation along a volcanic flank, etc.) The mean elevation change over a polygon $\Omega$ is:

$$\Delta z^{aggregate} = \frac{1}{N} \sum_{i=1}^{N} \Delta z_i$$

where the sum of $i$ individual topographic change measurements ($\Delta z_i$) is taken over the $N$ cells that fall inside $\Omega$.

<h3><a id="uncertainty_matters"></a>Why uncertainty matters</h3> 

The elevation change observed in a differenced raster can be decomposed as:

$$\Delta_{z}^{measured} = \Delta_{z}^{actual} + \Delta_{z}^{error}$$

where:
- **Δz<sub>actual</sub>** is the true vertical change (erosion, deposition, uplift, subsidence, construction, vegetation change, etc.)
- **Δz<sub>error</sub>** is the cumulative vertical error from data collection, processing, and alignment (GNSS/INS errors, boresight errors, DEM interpolation artifacts, alignment errors, metadata errors)

Errors present in either dataset are often of similar magnitude to the true change, particularly for legacy datasets acquired with less advanced instrumentation, lower point density, or incomplete metadata ([Glennie et al., 2014](https://doi.org/10.1002/2014GL059919)).


<h3><a id="error_types"></a>Error types in lidar topographic differencing</h3>

<h4><a id="short_scale"></a>Short-scale errors (meters to tens of meters)</h4>

| Error Type | Description | Key References |
|------------|-------------|----------------|
| **Random noise** | Sensor sensitivity, environmental conditions, and GNSS/IMU interference produce scattered positive/negative differences | [Glennie, 2007](https://doi.org/10.1515/jag.2007.017) |
| **Point misclassification** | Incorrect ground/vegetation/building classification creates meter-to-decameter artifacts with diffuse boundaries | [Passalacqua et al., 2015](https://doi.org/10.1016/j.earscirev.2015.05.012) |
| **Geometric distortion** | High laser incidence angles on steep slopes spread pulse energy, degrading measurement quality | [Schaer et al., 2007](https://doi.org/10.1117/12.717277) |


<h4><a id="mid_scale"></a>Mid-scale errors (hundred-meter scale)</h4> 

| Error Type | Description | Key References |
|------------|-------------|----------------|
| **Horizontal alignment errors** | Georeferencing offsets produce apparent vertical change correlated with topographic aspect; correctable via ICP registration | [Glennie et al., 2014](https://doi.org/10.1002/2014GL059919); [Besl & McKay, 1992](https://doi.org/10.1117/12.57955) |
| **Flight line striping** | Kinematic GNSS atmospheric delays create linear bands (hundreds of meters to kilometers wide) perpendicular to flight path | [Shan et al., 2007](https://doi.org/10.1201/9781420051438); [DeLong et al., 2022](https://doi.org/10.1029/2022EA002420) |

<h4><a id="long_scale"></a>Long-scale errors (kilometer scale)</h4>

| Error Type | Description | Key References |
|------------|-------------|----------------|
| **Instrument calibration biases** | Range calibration, uniform GNSS/IMU misalignment, or atmospheric corrections affect entire dataset uniformly | [Glennie, 2007](https://doi.org/10.1515/jag.2007.017); [Habib et al., 2009](https://doi.org/10.14358/PERS.75.10.1159) |
| **Geoid model errors** | Wrong geoid model produces 10–20 cm vertical shifts across the scene | Brigham et al. |
| **Ellipsoidal/orthometric confusion** | Mixing height systems causes vertical errors of tens of meters | Brigham et al. |


<h3><a id="workflow"></a>Our workflow</h3>

This notebook implements a geostatistical approach to quantify uncertainty in topographic differencing. The key insight is that **uncertainty is often structured, scale-dependent, and dominated by mid- and long-range correlation**.

**Notebook steps:**

1. [**Setup**](#setup): Install dependencies and configure the environment
2. [**Data Loading & Preprocessing**](#data-access): Import user-provided point clouds, extract and verify/update CRS metadata
3. [**CRS Transformation**](#CRS_transformation): Align coordinate reference systems, vertical datums, and epochs
4. [**Point Cloud Alignment**](#alignment): ICP co-registration to correct horizontal offsets between surveys
5. [**2D DEM Differencing**](#differencing): Generate DEMs from point clouds and compute pixel-by-pixel elevation change
6. [**Visualization**](#visualization): Plot DEMs, hillshades, slopes, and the difference raster
7. [**Stable Area Identification**](#define_stable_areas): Delineate control zones where no real change is expected
8. [**Descriptive Statistics**](#descriptive_stats): Characterize the error distribution and check stationarity assumptions
9. [**Systematic Error Estimation**](#estimate-error): Estimate and remove vertical bias using the median of stable-area differences
10. [**Variography**](#Variography): Fit nested variogram models to capture multi-scale spatial error structure
11. [**Uncertainty Propagation**](#uncertainty_propagation): Propagate the error model to features of interest via Monte Carlo integration

In [1]:
%matplotlib qt

<h2><a id="setup"></a>2. Setup</h2>

<h3><a id="colab"></a>Running the notebook in Colab</h3>

For ease-of-use, it is suggested to launch and execute these notebooks on <a href="https://colab.research.google.com/">Google Colaboratory</a> (Colab, for short), Google's Cloud Platform. Dependencies will be installed on a virtual machine on Google's cloud servers and the code will be executed directly in your browser. A major benefit of this is that you will have direct access to Google's high-end CPU/GPUs and will not have to install any dependencies locally. All deliverables will be saved to your personal Google Drive. To experiment and run one of the below Jupyter Notebooks on Google Colab click the "Open in Colab" badge below.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cassandra-Brigham/topographic-difference-uncertainty/blob/main/differencing_workflow_user_pointclouds.ipynb)

In [2]:
import os, sys, pathlib

# --- Colab guard ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1) Mount Drive (idempotent)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # 2) Check if condacolab is already fully configured
    # We verify both that conda exists AND that LD_LIBRARY_PATH is set
    conda_ready = (
        os.path.exists("/usr/local/bin/conda") and
        "LD_LIBRARY_PATH" in os.environ
    )
    
    if conda_ready:
        import condacolab
        condacolab.check()
        print("âœ“ Condacolab already installed and configured")
    else:
        print("Installing condacolab (kernel will restart)...")
        !pip install -q condacolab
        import condacolab
        condacolab.install()  # This restarts the kernel
else:
    print("Not running in Colab; skipping condacolab setup.")

Not running in Colab; skipping condacolab setup.


**Kernel Restart Required**

If this is your first time running this notebook, the cell above will have installed `condacolab` and **automatically restarted the kernel**. This is expected behavior.

**Please proceed by running the cell below** to continue with the environment setup. The next cell will install PDAL and the remaining dependencies.

<h3><a id="params"></a>Your parameters</h3>

Set your data path here!

- **Google Colab users**: Update `DATA_PATH` in the cell below to point to your data folder on Google Drive (e.g., `/content/drive/MyDrive/lidar-project`). Your Drive will be mounted when you run the setup cells.
- **Local users**: Update `DATA_PATH` in the cell below to point to your local data folder (e.g., `/Users/yourname/Documents/lidar-project`).

In [3]:
# Update your data path here

DATA_PATH = "test_data/paper_examples/ca_fires/pcs"#"your/path/here"
COMPARE_PC_NAME = "compare.laz"
REFERENCE_PC_NAME = "reference.laz"

# Set base data directory based on environment
from pathlib import Path
if IN_COLAB:
    BASE_DATA_DIR = Path(DATA_PATH)
    
    print(f"Using Colab data directory: {BASE_DATA_DIR}")
else:
    BASE_DATA_DIR = Path(DATA_PATH)
    
    print(f"Using local data directory: {BASE_DATA_DIR}")

# Create base directory if it doesn't exist
BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Helper function to resolve data paths
def get_data_path(*path_parts):
    return str(BASE_DATA_DIR / Path(*path_parts))

Using local data directory: test_data/paper_examples/ca_fires/pcs


Sometimes the point cloud file will lack important metadata or the metadata will be parsed incorrectly. To remedy this, we can manually update the point cloud metadata if we know more information. Fill out this information to the best of your ability. This information can be found in several places:

- **OpenTopography data pages**: Each dataset has a metadata page with survey details including CRS, datum, survey date, and geoid information. Look under the "Overview" or "Coordinates and Classification" sections. The "Full Metadata" link in the "Overview" section will take you to the original data index file.
- **Survey reports**: Most lidar acquisitions include a project report (often a PDF) documenting collection parameters, coordinate systems, and vertical datums.
- **USGS metadata**: For USGS 3DEP and other national elevation data, metadata is available at the [USGS National Geospatial Technical Operations Center](https://thor-f5.er.usgs.gov/ngtoc/metadata/waf/elevation/)
- **NOAA Digital Coast**: For coastal lidar data, metadata can be found in the [NOAA Digital Coast portal](https://coast.noaa.gov/digitalcoast/) or the individual dataset index files, e.g., [NOAA NOS Coastal Lidar index](https://noaa-nos-coastal-lidar-pds.s3.amazonaws.com/laz/geoid18/8866/index.html)
- **State/agency GIS portals**: Many state/international agencies maintain their own lidar portals with detailed metadata documentation.


In [6]:
# Complete these fields if known, set to None if unknown

# GEOID_COMPARE = None # e.g., "geoid12b"
# EPOCH_COMPARE = None #e.g., "05/18/2005 - 05/27/2005" or "05/22/2025"
# HORIZ_CRS_COMPARE = None # e.g., "32611" 
# VERT_CRS_COMPARE = None # e.g., "5703" 
# COMPLETE_CRS_COMPARE = None # e.g., "EPSG:3136+EPSG:5703"

# GEOID_REFERENCE = None 
# EPOCH_REFERENCE = None
# HORIZ_CRS_REFERENCE = None
# VERT_CRS_REFERENCE = None
# COMPLETE_CRS_REFERENCE = None

GEOID_COMPARE = "geoid18" 
EPOCH_COMPARE = "01/08/2023 - 01/07/2024"
HORIZ_CRS_COMPARE = "32611" 
VERT_CRS_COMPARE = "5703" 
COMPLETE_CRS_COMPARE = None
UNITS_COMPARE = "meter" 


GEOID_REFERENCE = "geoid18" 
EPOCH_REFERENCE ="01/01/2010"
HORIZ_CRS_REFERENCE = "6340"
VERT_CRS_REFERENCE = "5703"
COMPLETE_CRS_REFERENCE = None
UNITS_REFERENCE = "meter" 




<h3><a id="setup2"></a>Continue setup</h3>

In [7]:
import os, sys, pathlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Verify condacolab and install PDAL
    import condacolab
    condacolab.check()
    
    # Remove conflicting Python pin and install PDAL
    !rm -f /usr/local/conda-meta/pinned
    !mamba install -y -c conda-forge pdal python-pdal
    
    # Fix SQLite symlink - conda installs newer SQLite but old symlink remains
    # Dynamically find the installed SQLite version instead of hardcoding
    !sqlite_lib=$(ls /usr/local/lib/libsqlite3.so.3.* 2>/dev/null | head -1) && \
        if [ -n "$sqlite_lib" ]; then \
            sudo ln -sf "$sqlite_lib" /usr/local/lib/libsqlite3.so.0; \
            echo "Linked $sqlite_lib -> libsqlite3.so.0"; \
        fi
    
    # Fix numpy version conflict
    !{sys.executable} -m pip install -q "numpy<2.2"
    
    # Set PROJ environment
    os.environ['PROJ_LIB'] = '/usr/local/share/proj/'
    
    # Install topochange from GitHub
    print("\nInstalling topochange package...")
    !{sys.executable} -m pip install -q --no-cache-dir git+https://github.com/Cassandra-Brigham/topochange.git
    
    # Install additional packages not in topochange dependencies
    !{sys.executable} -m pip install -q small-gicp colormaps boto3
    
    # Verify PDAL via wrapper
    from topochange.pdal_wrapper import pdal, get_pdal_status
    status = get_pdal_status()
    print(f"\nEnvironment ready!")
    print(f"  PDAL version: {status['version']}")
    print(f"  PDAL mode: {status['mode']}")
else:
    print("Not running in Colab; skipping environment setup.")

Not running in Colab; skipping environment setup.


In [8]:
# Install visualization libraries
%pip install -q comm ipywidgets
%pip install -q ipyleaflet

# Fix pyproj PROJ path (Colab-specific, may need adjustment based on conda installation)
import os
import sys

# Set PROJ_LIB if needed (conda usually handles this automatically)
if IN_COLAB and not os.environ.get("PROJ_LIB"):
    from google.colab import output
    output.enable_custom_widget_manager()
    # Try common conda locations first
    possible_proj_paths = [
        "/opt/conda/share/proj",
        "/usr/share/proj", 
        "/usr/local/share/proj"
    ]
    for proj_path in possible_proj_paths:
        if os.path.isdir(proj_path):
            os.environ["PROJ_LIB"] = proj_path
            break

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


<h3><a id="import-libraries"></a>Import required libraries</h3>

This cell imports the necessary Python libraries for data handling and analysis, including custom functions from the provided scripts for differencing, variography, and stable area analysis.

In [9]:
import numpy as np
import rasterio
from scipy.spatial.transform import Rotation

# Core classes
from topochange import (
    Raster,
    PointCloud,
    PointCloudPair,
    VariogramAnalysis,
    RasterDataHandler,
    StatisticalAnalysis,
    RegionalUncertaintyEstimator,
    VariogramModelSelector,
    FittedVariogramModel,
    MODEL_REGISTRY,
    CompositeVariogramModel,
)

# Interactive features (requires ipyleaflet)
from topochange import (
    TopoMapInteractor,
    StableAreaRasterizer,
    StableAreaAnalyzer,
)

# Geoid utilities
from topochange.geoid_utils import (
    select_geoid_grid,
    ensure_proj_grids_for_region,
    get_all_proj_data_dirs,
)

# Alignment
from topochange.alignment import LandscapeAligner, RegistrationConfig

SEED = 42

# Ensure PROJ geoid grids are available (Colab only)
if IN_COLAB:
    print("Setting up PROJ geoid grids for Colab...")
    print(f"PROJ data directories: {get_all_proj_data_dirs()}")
    ensure_proj_grids_for_region('us_noaa', verbose=True)
    print("\nPROJ geoid grids ready")

<h2><a id="data-access"></a>3. Data access, download and pre-processing</h2>

This section covers loading your compare (older) and reference (newer) topographic **point clouds**. 

Use this notebook if you have local `.las` or `.laz` point cloud files. The code will generate Digital Terrain Models (DTMs) and Digital Surface Models (DSMs) from your data and then align them. If you are starting with your own DEMs, go to [this notebook](). If you are not providing your own point clouds or DEMs and want to search for and download data, go to [this notebook](). 

<h3><a id="upload-laz"></a>Upload your point cloud files</h3>

In [10]:
# Define point cloud file paths

compare_pc_path = get_data_path(COMPARE_PC_NAME)
reference_pc_path = get_data_path(REFERENCE_PC_NAME)

<h3><a id="extract_metadata"></a>Extract metadata from point cloud file</h3>

- Input format: use a point cloud in .las or .laz format.
- The path to the compare point cloud is already set in compare_pc_path.
- In the next cells, a PointCloud object (pc1) is created from compare_pc_path, its metadata is read from the file header/VLRs, and then printed for review.
- This step only inspects metadata (CRS, vertical datum/geoid, epoch, units, scales/offsets, bounds, classifications, returns, GPS time type); it does not modify the data.
- If key fields are missing or incorrect (e.g., compound CRS, horizontal/vertical CRS, geoid model, epoch), the following sections demonstrate how to update them using add_metadata before creating DEMs or performing transformations.


In [11]:
# Create PointCloud object and read in metadata
pc1 = PointCloud(compare_pc_path)
pc1.from_file()

In [12]:
pc1.print_metadata()


--- CRS Information ---
Property                  Value                                             
---------------------------------------------------------------------------
Compound CRS              PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",D
Horizontal CRS            PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",D
Vertical CRS              
Geoid model               None

--- Point Cloud Metadata ---
Property                  Value                                             
---------------------------------------------------------------------------
Total points              342,201,386
Bounds                    (403991.77, 3780732.16, 408021.32, 3783465.05)
Horizontal units          metre (m) (EPSG:9001)
Vertical units            unknown (?)

--- Time Information ---
Property                  Value                                             
---------------------------------------------------------------------------
Creation year             2026
Creation DOY            

<h3><a id="update_metadata"></a>Update point cloud metadata</h3>

<h4><a id="update_compound"></a>Update compound Coordinate Reference System</h4>

In [13]:
# Update compound CRS, using either a PROJ string, a WKT string, or an EPSG code.
# Skip this step if you are sure the compound CRS is correct

# If compound CRS is wrong or missing, can update metadata manually
  
if COMPLETE_CRS_COMPARE:
    pc1.add_metadata(compound_CRS=COMPLETE_CRS_COMPARE)
else:
    pass
print(f"Original compound CRS: {pc1.original_compound_crs}")  
print(f"Updated compound CRS: {pc1.current_compound_crs}")

Original compound CRS: PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32611"]]
Updated compound CRS: PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0]

<h4><a id="update_horiz"></a>Update horizontal Coordinate Reference System</h4>

In [14]:
# Update horizontal CRS, using either a PROJ string, a WKT string, or an EPSG code.
# Skip this step if you are sure the horizontal CRS is correct.

if HORIZ_CRS_COMPARE:
    pc1.add_metadata(horizontal_CRS=HORIZ_CRS_COMPARE)
else:
    pass

print ("Original horizontal CRS:", pc1.original_horizontal_crs)
print("Original compound CRS:", pc1.original_compound_crs)
print("")
print(f"Updated horizontal CRS: {pc1.current_horizontal_crs}")
print(f"Updated compound CRS : {pc1.current_compound_crs}")

Original horizontal CRS: PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32611"]]
Original compound CRS: PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin"

<h4><a id="update_vertical"></a>Update vertical Coordinate Reference System</h4>

In [15]:
# Update vertical CRS, using either a PROJ string, a WKT string, or an EPSG code.

# Skip this step if you are sure the vertical CRS is correct.

if VERT_CRS_COMPARE:
    pc1.add_metadata(vertical_CRS=VERT_CRS_COMPARE)
else:
    pass

print ("Original vertical CRS:", pc1.original_vertical_crs)
print("Original compound CRS:", pc1.original_compound_crs)
print("")
print(f"Updated vertical CRS: {pc1.current_vertical_crs}")
print(f"Updated compound CRS : {pc1.current_compound_crs}")

Original vertical CRS: 
Original compound CRS: PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32611"]]

Updated vertical CRS: VERTCRS["NAVD88 height",VDATUM["North American Vertical Datum 1988"],CS[vertical,1],AXIS["gravity-related height (H)",up,LENGTHUNIT["metre",1]],USAGE[SCOPE["Geodesy, engineering survey, topographic mapping."],AREA["Mexico - onshore. United States (USA) - CONUS and Alaska - onshore - Alabama; Alaska; Arizona; Arkansas; California

<h4><a id="update_geoid"></a>Update geoid</h4>

In [16]:
# If vertical coordinates are orthometric and geoid is wrong or missing, can update metadata manually
print(f"Original geoid model: \n {pc1.geoid_model}")
print(f"Original vertical CRS: \n {pc1.current_vertical_crs}")
print(f"Are the vertical coordinates orthometric?  \n {pc1.is_orthometric}")

Original geoid model: 
 None
Original vertical CRS: 
 VERTCRS["NAVD88 height",VDATUM["North American Vertical Datum 1988"],CS[vertical,1],AXIS["gravity-related height (H)",up,LENGTHUNIT["metre",1]],USAGE[SCOPE["Geodesy, engineering survey, topographic mapping."],AREA["Mexico - onshore. United States (USA) - CONUS and Alaska - onshore - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming."],BBOX[14.51,172.42,71.4,-66.91]],ID["EPSG",5703]]
Are the vertical coordinates orthometric?  
 True


In [17]:
# Update geoid, using either a PROJ string, a WKT string, or an EPSG code.
if GEOID_COMPARE:
    geoid_grid, _ =select_geoid_grid(GEOID_COMPARE,verbose=True)
    
    # If the automatic selection is not the desired grid, you can specify the index of your desired grid in the list of matches. Uncomment the lines below and set choice to the desired index.
    #geoid_grid_2, _ =select_geoid_grid(geoid, choice=1)
    #print(f"Select the second grid: {geoid_grid_2}")
    
    pc1.add_metadata(geoid_model=geoid_grid)
else:
    pass

print(f"Geoid model: {pc1.geoid_model}")
print(f"Vertical CRS:{pc1.current_vertical_crs}")
print(f"Are the vertical coordinates orthometric? {pc1.is_orthometric}")

Geoid grids for 'geoid18' (canonical 'geoid18'):
  [0] us_noaa_g2018u0.tif (local)  <== selected (auto-selected local u0/CONUS grid)
  [1] us_noaa_g2018p0.tif (local)
  [2] us_noaa_g2018u0.tif (local)
Geoid model: us_noaa_g2018u0.tif
Vertical CRS:VERTCRS["NAVD88 height",VDATUM["North American Vertical Datum 1988"],CS[vertical,1],AXIS["gravity-related height (H)",up,LENGTHUNIT["metre",1]],USAGE[SCOPE["Geodesy, engineering survey, topographic mapping."],AREA["Mexico - onshore. United States (USA) - CONUS and Alaska - onshore - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Wa

<h4><a id="update_epoch"></a>Update epoch</h4>

Accurate epoch metadata is critical when working with time-dependent coordinate reference systems and transformations.

- Many modern CRSs (e.g., ITRF, NAD83(2011), WGS84 realizations) are defined at a specific epoch and use velocities to propagate coordinates through time. Without the correct epoch, transformation pipelines cannot apply the right motion, leading to offsets.
- Horizontal positions can drift several centimeters per year due to tectonic motion; over a decade this can reach decimeter-level errors. Vertical biases can be similar or larger in areas with subsidence or uplift.
- When aligning point clouds to reference datasets collected at different times, mismatched epochs cause systematic misalignment that looks like a rigid shift or tilt rather than random noise.
- Velocity/deformation models and time-dependent Helmert transforms require an epoch to compute displacement; defaults (or missing epoch) often introduce hidden biases.
- Correct epoch improves repeatability, change detection, and metadata transparency, ensuring others can reproduce transformations and understand residuals.
- If data were collected over a range of dates, using the midpoint as the epoch is a practical, documented approximation for most transformations.

In [18]:
# If epoch is wrong or missing, can update metadata
print(f"Original epoch: {pc1.epoch}")

Original epoch: 2023.7634737125825


In [19]:
if EPOCH_COMPARE:
    # Update epoch, using either a decimal year, a single date, or a date range (the epoch will be set to the midpoint of the date range)
    pc1.add_metadata(epoch=EPOCH_COMPARE)
else:
    pass

print(f"Updated epoch: {pc1.epoch}")

Updated epoch: 2023.5177857624074


In [20]:
# Final check of all metadata
print("Current compound CRS:", pc1.current_compound_crs)
print("Current horizontal CRS:", pc1.current_horizontal_crs)
print("Current vertical CRS:", pc1.current_vertical_crs)
print("Current geoid model:", pc1.geoid_model)
print("Current epoch:", pc1.epoch)

Current compound CRS: COMPOUNDCRS["WGS 84 / UTM zone 11N + NAVD88 height",PROJCRS["WGS 84 / UTM zone 11N",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["UTM zone 11N",METHOD["Transverse Mercator",ID["EPSG",9807]],PARAMETER["Latitude of natural origin",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8801]],PARAMETER["Longitude of natural origin",-117,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8802]],PARAMETER["Scale factor at natu

<h3><a id="import_ref_cloud"></a>Import reference point cloud</h3>

In [21]:
# Create PointCloud object and read in metadata
pc2 = PointCloud(reference_pc_path)
pc2.from_file()

In [22]:
pc2.print_metadata()


--- CRS Information ---
Property                  Value                                             
---------------------------------------------------------------------------
Compound CRS              COMPD_CS["NAD83(2011) / UTM zone 11N + NAVD88 he
Horizontal CRS            PROJCS["NAD83(2011) / UTM zone 11N",GEOGCS["NAD8
Vertical CRS              VERT_CS["NAVD88 height",VERT_DATUM["North Americ
Geoid model               None

--- Point Cloud Metadata ---
Property                  Value                                             
---------------------------------------------------------------------------
Total points              359,678,568
Bounds                    (403351.38, 3779784.57, 408129.25, 3785032.3)
Horizontal units          metre (m) (EPSG:9001)
Vertical units            metre (m) (EPSG:9001)

--- Time Information ---
Property                  Value                                             
--------------------------------------------------------------------------

In [23]:
pc2.add_metadata(
    horizontal_CRS=HORIZ_CRS_REFERENCE if HORIZ_CRS_REFERENCE else None,
    vertical_CRS=VERT_CRS_REFERENCE if VERT_CRS_REFERENCE else None,
    compound_CRS=COMPLETE_CRS_REFERENCE if COMPLETE_CRS_REFERENCE else None,
    geoid_model=GEOID_REFERENCE if GEOID_REFERENCE else None,
    #epoch=EPOCH_REFERENCE if EPOCH_REFERENCE else None,
                 )

In [24]:
print("Current compound CRS:", pc2.current_compound_crs)
print("Current horizontal CRS:", pc2.current_horizontal_crs)
print("Current vertical CRS:", pc2.current_vertical_crs)
print("Current geoid model:", pc2.geoid_model)
print("Current epoch:", pc2.epoch)


Current compound CRS: COMPOUNDCRS["NAD83(2011) / UTM zone 11N + NAVD88 height",PROJCRS["NAD83(2011) / UTM zone 11N",BASEGEOGCRS["NAD83(2011)",DATUM["NAD83 (National Spatial Reference System 2011)",ELLIPSOID["GRS 1980",6378137,298.257222101,LENGTHUNIT["metre",1]],ANCHOREPOCH[2010]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",6318]],CONVERSION["UTM zone 11N",METHOD["Transverse Mercator",ID["EPSG",9807]],PARAMETER["Latitude of natural origin",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8801]],PARAMETER["Longitude of natural origin",-117,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8802]],PARAMETER["Scale factor at natural origin",0.9996,SCALEUNIT["unity",1],ID["EPSG",8805]],PARAMETER["False easting",500000,LENGTHUNIT["metre",1],ID["EPSG",8806]],PARAMETER["False northing",0,LENGTHUNIT["metre",1],ID["EPSG",8807]]],CS[Cartesian,2],AXIS["(E)",east,ORDER[1],LENGTHUNIT["metre",1]],AXIS["(N)",north,ORDER[2],LENGTHUNIT["metre",1]],USAGE[SCOPE["Engineering sur

<h3><a id="CRS_transformation"></a>Coordinate Reference System transformation</h3>

Before differencing, both datasets must share a common reference frame: not just the same map projection, but also the same vertical datum, geoid model, and coordinate epoch. Mismatches in any of these components introduce systematic errors that look like real elevation change.

| Component | What it defines | Error if mismatched |
|-----------|-----------------|---------------------|
| **Horizontal CRS** | Map projection and geodetic datum (e.g., UTM Zone 18N, NAD83) | Lateral shifts causing aspect-correlated vertical errors |
| **Vertical datum** | Height reference surface (ellipsoidal vs. orthometric) | Tens of meters if ellipsoid/orthometric confused |
| **Geoid model** | Equipotential surface relating ellipsoid to orthometric heights (e.g., GEOID12B, GEOID18) | 10–20 cm systematic offset |
| **Epoch** | Time of coordinate realization | cm/year drift from tectonic motion |

**The transformation pipeline:**

The `RasterPair.transform_raster1_to_match_raster2()` method applies transformations in a specific order:

1. **Epoch transformation** → Propagates coordinates through time using velocity models to account for tectonic plate motion. Critical when surveys span multiple years.

2. **Horizontal reprojection** → Converts between map projections and geodetic datums. Uses interpolation (bilinear recommended) to resample the grid.

3. **Vertical datum transformation** → Converts between ellipsoidal and orthometric heights, or between different geoid models. Applied as a Z-value adjustment.

4. **Grid alignment** → Resamples to match the target raster's exact pixel grid for cell-by-cell differencing.


**Checking for mismatches:**

Use `RasterPair.check_all_match()` or `PointCloudPair.check_all_match()` to identify which transformations are needed:
```python
comparison = raster_pair.check_all_match()
print(comparison['transformations_needed'])  # e.g., ['epoch', 'horizontal_crs', 'geoid']
```

**Common pitfalls:**

- Metadata may be incorrect or missing. Always verify CRS information against acquisition reports
- NAVD88 heights referenced to different geoid models (GEOID09, GEOID12B, GEOID18) are *not* directly comparable
- Legacy datasets often lack epoch information; use the acquisition date midpoint as a practical approximation

In [25]:
# Create PointCloudPair
pc_pair = PointCloudPair(pc1, pc2)

In [26]:
# Check what needs transformation
pc_pair.print_comparison()



--- PointCloudPair Comparison ---

Compare (pc1):   compare.laz
Reference (pc2): reference.laz

Parameter            Match    PC1                  PC2                 
----------------------------------------------------------------------
Horizontal CRS       No       EPSG:32611           EPSG:6340           
Vertical CRS         Yes      Orthometric          Orthometric         
Geoid Model          Yes      us_noaa_g2018u0.tif  us_noaa_g2018u0.tif 
Epoch                No       2023.5178            2025.1395           
Vertical Units       Yes      meter                meter               
----------------------------------------------------------------------

Transformations needed: horizontal_crs, epoch



In [27]:
# Transform compare to match reference (CRS + epoch)

transformed = pc_pair.transform_compare_to_match_reference(
    skip_epoch=True,
    skip_horizontal=False,
    skip_vertical=False,
    verbose=True
)

# Verify the transformed file
print(f"\nTransformed file: {transformed.filename}")

# Crop both clouds to overlap using the new transformed cloud

pc1_cropped, pc2_cropped = pc_pair.crop_to_overlap(
    use_transformed=True,
    interior_buffer=10.0,  # 10m buffer for alignment
    overwrite=True,
    verbose=True
)

# Verify both cropped files have similar bounds
print(f"\nCropped compare bounds: X=[{pc1_cropped.minx:.1f}, {pc1_cropped.maxx:.1f}], Y=[{pc1_cropped.miny:.1f}, {pc1_cropped.maxy:.1f}]")
print(f"Cropped reference bounds: X=[{pc2_cropped.minx:.1f}, {pc2_cropped.maxx:.1f}], Y=[{pc2_cropped.miny:.1f}, {pc2_cropped.maxy:.1f}]")


--- Transform compare to match reference ---
Transformations needed: ['horizontal_crs', 'epoch']
  Horizontal CRS reprojection needed
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)
  Combined transformation [done]

Output: test_data/paper_examples/ca_fires/pcs/compare_reproj_transformed.laz

--- Computing true data footprint for reference cloud ---



Transformed file: test_data/paper_examples/ca_fires/pcs/compare_reproj_transformed.laz


True extent computed successfully
True extent bounds (utm): (403343.92000049923, 3779781.5401733126, 408132.09975889814, 3785035.8001042013)
True extent area (utm): 20,287,503 m²
PC2 header bounds: (403351.38, 3779784.57) to (408129.25, 3785032.30)

--- Cropping to Overlap Area ---
PC1 poly_4326 bounds: (-118.04183156436567, 34.16324391188875, -117.99797015581036, 34.18791672031423)
PC2 poly_4326 bounds: (-118.04925331277612, 34.15447347350021, -117.99691608980231, 34.202273716701846)
Common CRS: 6340
PC1 native CRS: PROJCRS["NAD83(2011) / UTM zone 11N",BASEGEOGCRS["NAD83(2011)",DATUM["NAD83 (Nat...
PC2 native CRS: PROJCRS["NAD83(2011) / UTM zone 11N",BASEGEOGCRS["NAD83(2011)",DATUM["NAD83 (Nat...
PC1 area (bounding box): 10,935,005 m²
PC2 area (bounding box): 20,493,202 m²
PC2 area (true extent): 20,287,503 m²
Intersection area: 9,918,787 m²
Overlap fraction pc1: 90.7%
Overlap fraction pc2: 48.4%
Interior buffer for compare: 10.0 m
Using true extent for buffer: True
PC2 extent used fo


Cropped compare bounds: X=[403991.8, 408019.2], Y=[3780746.1, 3783465.0]
Cropped reference bounds: X=[403983.3, 408038.2], Y=[3780727.5, 3783481.3]


Cropping complete.


<h3><a id="alignment"></a>Point cloud alignment</h3>

Before differencing, the two point clouds must be spatially aligned. Even small horizontal offsets between surveys produce apparent vertical changes that correlate with topographic aspect. Co-registration corrects these offsets by finding the optimal transformation that minimizes differences between overlapping stable terrain.

**Registration methods:**

The `LandscapeAligner` is built off of [small-gicp](https://github.com/koide3/small_gicp) and implements several Iterative Closest Point (ICP) variants:

| Method | Description | Best for |
|--------|-------------|----------|
| `icp` | Classic point-to-point ICP | Simple, fast alignment |
| `plane_icp` | Point-to-plane ICP | Smoother surfaces |
| `gicp` | Generalized ICP with covariance | Robust to noise, preferred for local use |
| `vgicp` | Voxelized GICP (GPU-accelerated) | Large point clouds, preferred for Colab use |

**Key parameters:**

- **`max_correspondence_distance`**: Maximum distance (meters) to consider point pairs as correspondences. Too small misses valid pairs; too large includes erroneous matches. Start with ~1 m for typical lidar.

- **`crop_dimensions`**: Cropping to a smaller region (e.g., 200×200 m) speeds computation and focuses alignment on a well-characterized area. The resulting transformation is then applied to the full dataset.

- **`point_filter="ground"`**: Using only ground-classified points avoids alignment errors from vegetation differences between surveys. If you want more classes than ground, you can provide a list of the class numbers you want to include (e.g. [2,6] for ground points and buildings.)


**Interpreting results:**

- **Fitness score**: Fraction of source points with valid correspondences (0–1). Values >0.5 typically indicate good alignment.
- **RMSE**: Root-mean-square error of point-to-point distances after alignment. Lower is better; values <0.1 m indicate excellent registration.
- **Transformation matrix**: The 4×4 matrix encoding the translation (and rotation if enabled) applied to align source → target.

After successful alignment, apply the transformation to your source point cloud before proceeding to DEM generation and differencing.

In [ ]:
if IN_COLAB:
    config = RegistrationConfig(
        point_filter="ground",
        max_correspondence_distance=1.0,
        crop_dimensions=(200, 200),
        method="vgicp",
    )
else:
    config = RegistrationConfig(
        point_filter="ground",
        max_correspondence_distance=1.0,
        #crop_dimensions=(500, 500),
        method="vgicp",
    )

aligner = LandscapeAligner(config)
result = aligner.align(source=pc1_cropped, target=pc2_cropped, apply_transform=True)

# # If you want to customize the registration parameters, you can create a RegistrationConfig object with your desired settings.
# # Uncomment and modify parameters as needed; defaults and parameter explanations shown below.

# config = RegistrationConfig(
#     # General parameters
#     method="gicp",  # str: "icp", "plane_icp", "gicp", "vgicp"
#     max_correspondence_distance=1.0,  # float or None: distance threshold (meters); None = auto-compute
#     max_iterations=50,  # int: maximum iterations for registration
    
#     # Centering and cropping
#     center_to_origin=True,  # bool: center both clouds to (0,0,0) before registration
#     crop_dimensions=None,  # tuple(float, float) or None: (x, y) crop rectangle in meters, centered on origin, or None to skip cropping
    
#     # Downsampling
#     downsample=False,  # bool: voxel grid downsampling using PDAL's filters.voxelcenternearestneighbor filter.
#     voxel_size=None,  # float or None: voxel size in meters; None = auto-compute when downsample=True, based on target_points
#     target_points=100000,  # int: target number of points after downsampling. if voxel_size is set, this is ignored.
    
#     # Coarse alignment
#     perform_coarse_alignment=True,  # bool: perform initial coarse alignment. If False, assumes clouds are roughly aligned already.
#     use_ground_plane_constraint=True,  # bool: If use_ground_plane_constraint=True (default for landscapes), only translation is applied, no rotation. This is appropriate for topographic data where both surveys should have the same "up" direction.
    
#     # Point filtering
#     point_filter="ground",  # str: "ground", "all", or "custom" (uses classification_filter)
#     use_ground_filter=False,  # bool: apply SMRF filter to classify ground (for unclassified data)
#     ground_filter_params={ # dict or None: SMRF parameters, e.g., {"cell": 1.0, "slope": 0.15, ...}
#         "cell": 1.0,       # Cell size (meters) for the grid used in morphological operations. Smaller values = finer detail but slower processing.
#         "scalar": 1.25,    # Elevation scalar for the progressive morphological filter. Controls how aggressively non-ground points are identified.
#         "slope": 0.15,     # Slope threshold (rise/run). Points with local slope exceeding this are candidates for non-ground classification.
#         "threshold": 0.5,  # Elevation threshold (meters). Maximum vertical distance a point can be from the ground surface to be classified as ground.
#         "window": 18.0,    # Maximum window size (meters) for morphological operations. Larger values handle bigger features (buildings, trees) but may smooth terrain.
#     }
    
#     # Outlier removal
#     outlier_removal=True,  # bool: remove statistical outliers before alignment
#     outlier_k_neighbors=20,  # int: number of neighbors for outlier detection
#     outlier_std_multiplier=2.0,  # float: standard deviations above mean to flag as outlier
    
#     # Classification filter (when point_filter="custom") ---
#     classification_filter=None,  # list[int] or None. E.g., [2] for ground, [2, 8] for ground + model key
    
#     # Validation thresholds
#     min_fitness_score=0.3,  # float: minimum acceptable fitness score (0-1). Fitness is the fraction of source points that found a valid correspondence in the target point cloud after alignment.
#     max_rmse=None,  # float or None: maximum acceptable RMSE; None: defaults to 5m
    
#     # Auto-retry on failure
#     enable_auto_retry=True,  # bool: retry with relaxed parameters if alignment fails
#     max_retries=3,  # int: maximum number of retry attempts
#     retry_strategies=[ # list[str]: strategies to try
#         "increase_correspondence", # x1.5 larger correspondence distance
#         "change_method",# on run 1 switch to vgicp if not already using it, on run 2 switch to icp
#         "adjust_filtering"], # relax outlier removal (outlier_std_multiplier Ã— 1.5) , use more points
# )

# aligner = LandscapeAligner(config)
# result = aligner.align(source=pc1_cropped, target=pc2_cropped, apply_transform=True)

INFO:topochange.alignment:Starting registration: compare_reproj_transformed_intersection_buffered -> reference_intersection
INFO:topochange.alignment:Filtered source to classifications: [2]
INFO:topochange.alignment:Removed outliers from source
INFO:topochange.alignment:Filtered target to classifications: [2]
INFO:topochange.alignment:Removed outliers from target
INFO:topochange.alignment:After preprocessing: 54074007 source, 54798906 target points
INFO:topochange.alignment:Centering to origin (centroid: [405646.7, 3782093.5, 440.9])
INFO:topochange.alignment:Coarse alignment completed
INFO:topochange.alignment:Running vgicp registration...
INFO:topochange.alignment:Preprocessing (covariance estimation, no downsampling)...


In [ ]:
print(f"RMSE: {result.rmse:.4f} m")
print(f"Fitness: {result.fitness:.2%}")
print(f"Converged: {result.converged}")
print(f"Iterations: {result.iterations}")
print(f"Method: {result.method_used}")
print(f"\nTransformation matrix:\n{result.transformation}")
print(f"\nCentroid used: {result.centroid}")

# Extract rotation (3x3) and translation (3x1) from 4x4 transformation
rotation = result.transformation[:3, :3]
translation = result.transformation[:3, 3]

print(f"Translation (x, y, z): {translation}")
print(f"Rotation matrix:\n{rotation}")

# Convert rotation matrix to Euler angles (in degrees)
r = Rotation.from_matrix(rotation)
euler_angles = r.as_euler('xyz', degrees=True)
print(f"Rotation (roll, pitch, yaw) i `n degrees: {euler_angles}")

RMSE: 0.2609 m
Fitness: 98.27%
Converged: True
Iterations: 32
Method: vgicp

Transformation matrix:
[[ 9.99999999e-01  3.64955272e-05 -2.38751387e-05 -4.65405324e-02]
 [-3.64949674e-05  9.99999999e-01  2.34478562e-05 -7.43404849e-02]
 [ 2.38759945e-05 -2.34469848e-05  9.99999999e-01  9.86311820e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

Centroid used: [3.54273636e+05 4.05406185e+06 1.31166241e+03]
Translation (x, y, z): [-0.04654053 -0.07434048  0.98631182]
Rotation matrix:
[[ 9.99999999e-01  3.64955272e-05 -2.38751387e-05]
 [-3.64949674e-05  9.99999999e-01  2.34478562e-05]
 [ 2.38759945e-05 -2.34469848e-05  9.99999999e-01]]
Rotation (roll, pitch, yaw) i `n degrees: [-0.00134341 -0.00136799 -0.00209101]


<h3><a id="differencing"></a>2D DEM differencing</h3>

Vertical differencing (also called raster subtraction or DoD—DEM of Difference) computes pixel-by-pixel elevation change between two co-registered DEMs. This is the fundamental operation for quantifying topographic change, producing a raster where each cell contains Δz = z₂ − z₁.

**Comparing processing scenarios:**

This example computes differences under three scenarios to illustrate how coordinate transformations and alignment affect results:

| Scenario | DEM Sources | What it tests |
|----------|-------------|---------------|
| **Horizontal-only** | Raw DSMs (no vertical transformation, same horizontal CRS) | Baseline; may include datum offsets |
| **Transformed** | After full 4D transformation (horizontal + vertical + epoch) | Effect of CRS transformation |
| **Aligned** | After transformation + ICP co-registration | Best-case scenario with geometric correction |

**Key parameters:**

- **`dem1`, `dem2`**: Specify which DEM products to difference. Options include `"dsm"`, `"dtm"`, `"dsm_transformed"`, `"dsm_transformed_aligned"`, etc.

- **`skip_epoch`**: If `True`, skips time-dependent coordinate transformations (e.g., plate motion corrections). Use to speed up process, when both datasets are already in the same epoch or when epoch differences are negligible.

**Interpreting the statistics:**

- **Mean/Median**: Non-zero values indicate systematic vertical bias. The median is more robust to outliers from real change or errors.

- **Standard deviation**: Quantifies the spread of elevation differences. High σ may indicate alignment issues, significant real change, or error sources like vegetation differences.

**What to look for in the difference maps:**

- *Aspect-correlated patterns* (positive on one slope aspect, negative on the opposite) → horizontal misalignment remains
- *Uniform offset across the scene* → vertical datum or calibration bias
- *Linear banding perpendicular to flight direction* → flight line errors ([Shan et al., 2007](https://doi.org/10.1201/9781420051438))
- *Localized clusters of change* → real geomorphic change or point misclassification

Comparing statistics across scenarios helps diagnose error sources: if alignment substantially reduces standard deviation, horizontal offsets were a dominant error. If the median shifts after transformation, vertical datum differences were present.

In [ ]:
# ============================================
# DIFFERENCING
# ============================================

output_dir = get_data_path("dem_output")
os.makedirs(output_dir, exist_ok=True)

# Scenario 1: Horizontal-only (NOT aligned)
results_horizontal_only = pc_pair.compute_2d_difference(
    dem1="dtm",
    dem2="dtm",
    overwrite=True,
    diff_output_path=os.path.join(output_dir, "diff_horizontal_only_dtm.tif"),
    verbose=True,
    classification_filter=None,
    #radius=5.0,
)
result_horizontal_only = results_horizontal_only['raster_pair']

# Scenario 2: Fully transformed (NOT aligned)
results_transformed = pc_pair.compute_2d_difference(
    dem1="dtm_transformed",
    dem2="dtm",
    skip_epoch=True,
    overwrite=True,
    diff_output_path=os.path.join(output_dir, "diff_transformed_dtm.tif"),
    verbose=True,
    classification_filter=None,
    #radius=5.0,
)
result_transformed = results_transformed['raster_pair']

# Scenario 3: Fully transformed + ICP aligned
# Now this will use the alignment we just computed!
results_aligned = pc_pair.compute_2d_difference(
    dem1="dtm_transformed_aligned",
    dem2="dtm",
    skip_epoch=True,
    overwrite=True,
    diff_output_path=os.path.join(output_dir, "diff_aligned_dtm.tif"),
    verbose=True,
    classification_filter=None,
    #radius=5.0,
)
result_aligned = results_aligned['raster_pair']


# Print stats
print(f"\n{'='*60}")
print(f"{'Scenario':<25} {'Mean':>10} {'Median':>10} {'Std':>10}")
print(f"{'='*60}")
print(f"{'Horizontal-only':<25} {results_horizontal_only['stats']['mean']:>10.4f} {results_horizontal_only['stats']['median']:>10.4f} {results_horizontal_only['stats']['std']:>10.4f}")
print(f"{'Transformed':<25} {results_transformed['stats']['mean']:>10.4f} {results_transformed['stats']['median']:>10.4f} {results_transformed['stats']['std']:>10.4f}")
print(f"{'Transformed+Aligned':<25} {results_aligned['stats']['mean']:>10.4f} {results_aligned['stats']['median']:>10.4f} {results_aligned['stats']['std']:>10.4f}")
print(f"{'='*60}")


Computing 2D (DEM-based) Difference

--- No horizontal transform needed, using original pc1 ---
DEM1 source: dtm (compare.laz)
DEM2 source: dtm (reference_intersection.laz)
DEM1 type: DTM
DEM2 type: DTM

Creating DEM from pc1: compare.laz
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)
Creating DEM from pc2: reference_intersection.laz

DEM1: test_data/paper_examples/az_high_slope/small/compare_dtm_1m.tif
DEM2: test_data/paper_examples/az_high_slope/small/reference_intersection_dtm_1m.tif

RasterPair comparison:

--- Computing Difference (raster2 - raster1) ---

--- Raster Transformation ---
Step                           Status                        
------------------------------


--- RasterPair Summary ---

Property           Raster1              Raster2             
----------------------------------------------------------
File               compare_dtm_1m.tif   reference_intersec  
Epoch              2017.3633056366493   2019.4931506849316  
Geoid              us_noaa_g2012bu0.t   us_noaa_g2012bu0.t  
Vertical Units     metre                unknown             

Parameter            Match   
----------------------------
Horizontal CRS       Yes     
Vertical CRS         Yes     
Geoid                Yes     
Epoch                No      
Units                Yes     
Grid                 No      

Transformations needed: epoch, grid



Grid alignment                 [done]
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)
------------------------------------------------------------

Source       Valid Pixels         Percent   
------------------------------------------
Raster1         1,483,867 / 1,601,234   92.7%
Raster2         1,518,524 / 1,601,234   94.8%
Overlap         1,474,212 / 1,601,234   92.1%

--- Difference Statistics ---
Statistic    Value (m)      
---------------------------
Min              -23.7430
Max               33.3145
Mean               1.0878
Std                0.5489
Median             1.0137
NMAD               0.1975
RMSE               1.2185

Output: test_data/paper_examples/az_high_slope


--- RasterPair Summary ---

Property           Raster1              Raster2             
----------------------------------------------------------
File               compare_dtm_1m.tif   reference_intersec  
Epoch              2017.3633056366493   2019.4931506849316  
Geoid              us_noaa_g2012bu0.t   us_noaa_g2012bu0.t  
Vertical Units     metre                unknown             

Parameter            Match   
----------------------------
Horizontal CRS       Yes     
Vertical CRS         Yes     
Geoid                Yes     
Epoch                No      
Units                Yes     
Grid                 No      

Transformations needed: epoch, grid




Source       Valid Pixels         Percent   
------------------------------------------
Raster1         1,482,039 / 1,601,234   92.6%
Raster2         1,518,524 / 1,601,234   94.8%
Overlap         1,472,443 / 1,601,234   92.0%

--- Difference Statistics ---
Statistic    Value (m)      
---------------------------
Min              -26.5682
Max               19.0023
Mean               0.9814
Std                0.3904
Median             0.9611
NMAD               0.0550
RMSE               1.0563

Output: test_data/paper_examples/az_high_slope/small/dem_output/diff_transformed_dtm.tif

Computing 2D (DEM-based) Difference

--- Auto-preparing: ICP alignment ---

--- Point Cloud Alignment (small_gicp) ---
Method: GICP
Using stored cropped compare cloud: test_data/paper_examples/az_high_slope/small/compare_intersection_buffered.laz
Using stored cropped reference cloud: test_data/paper_examples/az_high_slope/small/reference_intersection.laz
Downsample resolution: 0.5 m
Max correspondence distanc

--- LM optimization ---
iter=0 inner=0 e=1.20871e+07 new_e=1.44113e+06 lambda=0.001 dt=0.500551 dr=2.64192e-05
iter=1 inner=0 e=1.17107e+06 new_e=1.1666e+06 lambda=0.0001 dt=0.028818 dr=3.2875e-06
iter=2 inner=0 e=1.16659e+06 new_e=1.16659e+06 lambda=1e-05 dt=3.71901e-07 dr=1.1233e-10



Alignment Results:
  Converged: True
  Fitness (inlier ratio): 0.6235
  RMSE: 0.9938 m
  Inlier correspondences: 1,181,241
  Translation: [-0.0593, 0.1748, 0.4741] m
  Rotation: 0.0014°

Applying transformation to: test_data/paper_examples/az_high_slope/small/compare_transformed_intersection_buffered_aligned_transformed.laz
INFO:topochange.alignment_utils:Saved transformed point cloud to test_data/paper_examples/az_high_slope/small/compare_transformed_intersection_buffered_aligned_transformed.laz


DEM1 source: dtm_transformed_aligned (compare_transformed_intersection_buffered_aligned_transformed.laz)
DEM2 source: dtm (reference_intersection.laz)
DEM1 type: DTM
DEM2 type: DTM

Creating DEM from pc1: compare_transformed_intersection_buffered_aligned_transformed.laz
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https:


--- RasterPair Summary ---

Property           Raster1              Raster2             
----------------------------------------------------------
File               compare_transforme   reference_intersec  
Epoch              2017.3633056366493   2019.4931506849316  
Geoid              us_noaa_g2012bu0.t   us_noaa_g2012bu0.t  
Vertical Units     unknown              unknown             

Parameter            Match   
----------------------------
Horizontal CRS       Yes     
Vertical CRS         Yes     
Geoid                Yes     
Epoch                No      
Units                Yes     
Grid                 No      

Transformations needed: epoch, grid


Scenario                        Mean     Median        Std
Horizontal-only               1.0878     1.0137     0.5489
Transformed                   0.9814     0.9611     0.3904
Transformed+Aligned           0.5187     0.4916     0.4053



Source       Valid Pixels         Percent   
------------------------------------------
Raster1         1,459,920 / 1,601,234   91.2%
Raster2         1,518,524 / 1,601,234   94.8%
Overlap         1,459,920 / 1,601,234   91.2%

--- Difference Statistics ---
Statistic    Value (m)      
---------------------------
Min              -27.1772
Max               38.2549
Mean               0.5187
Std                0.4053
Median             0.4916
NMAD               0.0798
RMSE               0.6583

Output: test_data/paper_examples/az_high_slope/small/dem_output/diff_aligned_dtm.tif


In [29]:
# Plot the difference raster
fig = result_aligned.plot_difference(
    center_zero = True,
    vmin=-5,
    vmax=5,
)

fig = result_transformed.plot_difference(
    center_zero = True,
    vmin=-5,
    vmax=5,
)

fig = result_horizontal_only.plot_difference(
    center_zero = True,
    vmin=-5,
    vmax=5,
)

NameError: name 'result_aligned' is not defined

In [34]:
pair = result_aligned
results = results_aligned

<h2><a id="visualization"></a>4. Visualization and Derived Rasters</h2>

This section focuses on visualizing the results and creating derived topographic products like hillshades and slope maps, which are useful for interpreting the observed changes.

<h3><a id="plot-dems"></a>Plot the DEMs and derived rasters</h3>

In [31]:
# Generate derivatives for both rasters
hillshade1, hillshade2 = pair.generate_derivative("hillshade")
slope1, slope2 = pair.generate_derivative("slope")
aspect1, aspect2 = pair.generate_derivative("aspect")
roughness1, roughness2 = pair.generate_derivative("roughness")

# Plot side-by-side with automatic derivative generation
fig, axes = pair.plot_pair(derivative='dem')        # Original DEMs
fig, axes = pair.plot_pair(derivative='hillshade')  # Hillshades
fig, axes = pair.plot_pair(derivative='slope')      # Slopes
fig, axes = pair.plot_pair(derivative='aspect')     # Aspects
fig, axes = pair.plot_pair(derivative='roughness')  # Roughness

# Customize hillshade parameters
fig, axes = pair.plot_pair(
    derivative='hillshade',
    azimuth=270,    # West-facing light
    altitude=30,    # Low sun angle
    figsize=(14, 6)
)

/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


In [37]:
pair.plot_difference(vmin=-5,
    vmax=5,)

/Users/cassandrabrigham/ASU Dropbox/Cassandra Brigham/Mac/Documents/POSTDOC/Code/topographic-differencing-uncertainty/src/topochange/raster.py:1962: RuntimeWarning: invalid value encountered in divide
  interp = np.where(weight_sum > 0, weighted_sum / weight_sum, np.nan)
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


In [33]:
-

SyntaxError: invalid syntax (476313318.py, line 1)

<h2><a id="error_analysis"></a>5. Error analysis</h2>

<h3><a id="theoretical_framework"></a>Theoretical framework</h3>

The error analysis framework implemented here follows the geostatistical approach described by [Rolstad et al., 2009](https://doi.org/10.3189/002214309789470950) and [Hugonnet et al., 2022](https://doi.org/10.1109/JSTARS.2022.3188922), adapted for lidar-derived topographic differencing.

Topographic differencing errors can be decomposed into three components:

1. **Systematic vertical bias** (μ): A constant offset between datasets, typically estimated as the median of stable-area differences to reduce outlier influence

2. **Spatially correlated random error**: Errors that exhibit spatial structure due to:
   - Point misclassification (meter to decameter scale)
   - Geometric distortion on slopes (decameter scale)
   - Horizontal alignment errors (hundred-meter scale)
   - Flight line striping (hundred-meter to kilometer scale)
   - Vertical datum inconsistencies (kilometer scale)

3. **Uncorrelated random noise** (nugget): High-frequency noise from sensor sensitivity, environmental conditions, and surface reflectivity ([Glennie, 2007](https://doi.org/10.1515/jag.2007.017))

The semivariogram γ(h) quantifies how the variance of differences increases with separation distance h (Matheron, 1965; [Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277)):

$$\gamma(h) = \frac{1}{2N(h)} \sum_{i=1}^{N(h)} [\Delta_z(x_i) - \Delta_z(x_i + h)]^2$$

Key variogram parameters:
- **Nugget (c₀)**: Discontinuity at origin representing uncorrelated noise
- **Sill (c)**: Plateau value representing total variance
- **Range (a)**: Distance at which spatial correlation decays to negligible levels

For lidar data, **nested variograms** with multiple components capture error structures at different scales. Each spatial scale (short, medium, long range) contributes a portion of the total error variance.

<h3><a id="define_stable_areas"></a>Define stable areas</h3>


Stable areas are regions where **no topographic change is expected** between surveys. They serve as control zones for error calibration and the elevation differences within these areas represent pure error, allowing us to characterize the spatial structure of uncertainty.

<h4><a id="size_requirements"></a>Size requirements</h4>

The stable area must be large enough to reliably estimate the empirical variogram. Key constraints include (Journel & Huijbregts, 1978; [Oliver & Webster, 2015](https://doi.org/10.1007/978-3-319-15865-5)):

<h5><a id="min_pairs"></a>1. Minimum pairs per lag bin</h5>
Each lag bin in the variogram requires **at least 30 pairs**, with 50+ pairs recommended. Fewer than 20 pairs per bin produces unreliable estimates with high variance.

<h5><a id="var_coverage"></a>2. Variogram coverage rule</h5>
The variogram should span **less than half the domain size** to avoid pairing samples from opposite edges (Journel & Huijbregts, 1978). This means that if you expect correlation ranges up to 500 m, your stable area should have at least a 1 km extent in its longest dimension.

<h5><a id="corr_scales"></a>3. Capturing all correlation scales</h5> 
The largest correlation scale has the greatest impact on uncertainty estimates ([Rolstad et al., 2009](https://doi.org/10.3189/002214309789470950)). Typical lidar error correlation ranges span:

- **Short-range**: 10–100 m (point classification, geometric distortion)
- **Mid-range**: 100–1000 m (horizontal alignment, flight lines)
- **Long-range**: >1 km (vertical datum, geoid errors)

Your stable areas should extend far enough to capture all relevant scales.

<h4><a id="good_stable_area"></a>What makes a good stable area?</h4> 

**Ideal choices**:
- Roads and parking lots (paved, unvegetated)
- Bedrock outcrops (geologically stable)
- Flat, undisturbed terrain
- Areas with similar terrain properties (slope, roughness) to your areas of interest

**Areas to avoid**:
- Vegetation change zones (leaf-on vs. leaf-off)
- Construction or development areas
- Water bodies (variable water levels)
- Very steep slopes (higher geometric distortion)
- DEM boundary edges (edge effects)
- Areas with real expected change

<h4><a id="heteroscedasticity_consideration"></a>Heteroscedasticity consideration</h4>

Errors can vary with terrain properties. [Hugonnet et al., 2022](https://doi.org/10.1109/JSTARS.2022.3188922) explored how error variance changes with parameters such as slope and roughness. Ideally, stable areas should have similar terrain characteristics to your areas of interest, or you should account for this heteroscedasticity in the analysis. [xDEM](https://xdem.readthedocs.io/en/stable/uncertainty.html#heteroscedasticity) offers tools to model raster heteroscedasticity.



**Use the interactive map below to draw polygons over areas you consider stable.**

**Instructions:**
1. Use the polygon tool to draw areas of no expected change
2. Select "Stable" from the layer dropdown when drawing
3. Aim for areas totaling at least 1 km² if you expect long-range correlations
4. Run the cells below after drawing your polygons

In [38]:
diff = results.get("difference_raster")

In [52]:
out_folder_poly = Path.joinpath(BASE_DATA_DIR,"polygons/")
os.makedirs(out_folder_poly, exist_ok=True)


interactor = TopoMapInteractor(
    topo_diff_path=diff.filename,
    hillshade_path=hillshade1.filename,
    output_dir=out_folder_poly,
    overlay_dpi=600,
    overlay_vmin=-20,
    overlay_vmax=20,
)

interactor.map

/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)
Overlay: shape=(866, 1849), nodata=nan
Valid pixels: 1,459,920 / 1,601,234 (91.2%)
Color range: [-20.000, 20.000]
Data range: [-27.177, 38.255]
Saved overlay: test_data/paper_examples/az_high_slope/small/polygons/diff_aligned_dtm-d2b6fee480704e028d1e89a45da887a9.png
Generated data URL: 696278 chars (522190 bytes)
Map bounds (lat/lon): ((36.616900440075256, -112.63976616028745), (36.62498701981779, -112.61926015911651))
Map center: (36.62094372994652, -112.62951315970199)
Using overlay: test_data/paper_examples/az_high_slope/small/polygons/diff_aligned_dtm-d2b6fee480704e028d1e89a45da887a9.png


Map(center=[36.62094372994652, -112.62951315970199], controls=(ZoomControl(options=['position', 'zoom_in_text'…

In [53]:
interactor.stable_geoms

[<POLYGON ((353402.347 4054483.145, 353396.663 4053699.671, 355153.769 405363...>]

In [54]:
interactor.unstable_geoms

[]

<h3><a id="descriptive_stats"></a>Descriptive statistics</h3>

Before fitting a variogram, it is essential to examine the statistical characteristics of the stable area(s). Descriptive statistics provide a first-order assessment of the differencing errors and help evaluate whether the data meet the assumptions required for geostatistical analysis.

**Key statistics to examine:**

- **Mean and Median**: The median of the elevation differences in stable areas estimates the systematic vertical bias between the two DEMs. A non-zero median indicates a consistent offset that should be removed before variogram analysis. The mean is more sensitive to outliers, so comparing mean and median helps identify skewness in the distribution.

- **Standard Deviation and Variance**: These quantify the overall spread of elevation differences. High variance may indicate significant error sources or residual real change in areas assumed to be stable.

- **Skewness and Kurtosis**: Departures from normality can signal issues. Positive skewness might indicate unremoved vegetation or construction, while heavy tails (high kurtosis) could reflect outliers from misclassification or edge effects.

- **Percentiles (0.5%, 99.5%)**: Examining extreme percentiles helps identify outliers that may need to be filtered before variogram estimation.

**Assessing stationarity across stable areas:**

A fundamental assumption in geostatistics is *stationarity* — that the mean and variance of the error field are constant across the study area, and that spatial covariance depends only on separation distance, not absolute location ([Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277)). Examining the statistics of each stable area *separately* provides a practical check:

1. **Compare means/medians across areas**: Substantially different median values may indicate spatially varying bias (e.g., from flight line effects or tilted datums), violating the assumption that a single variogram model applies everywhere.

2. **Compare variances across areas**: If one stable area has much higher variance than another, the error characteristics may depend on location or terrain properties. This *heteroscedasticity* suggests that a single variogram may not adequately represent error structure across the entire scene ([Hugonnet et al., 2022](https://doi.org/10.1109/JSTARS.2022.3188922)).

3. **Look for systematic patterns**: If stable areas on one side of the scene consistently show positive differences while those on the other side show negative differences, a regional trend or tilt may be present that should be modeled separately.

The table and histogram below summarize the distribution of elevation differences in your stable area(s). If you have defined multiple stable polygons, compare their individual statistics to assess spatial consistency.

In [55]:
stable_polys, _ = interactor.export_geodataframes()

# one combined mask
rasterizer_stable = StableAreaRasterizer(interactor.topo_diff.path, stable_polys, nodata=-9999)
analyzer_stable = StableAreaAnalyzer(rasterizer_stable)

# Combined-area stats
df_all_stable_polys = analyzer_stable.stats_all(Path.joinpath(BASE_DATA_DIR,"polygons/combined_stable.tif"))

# Per-area stats
df_each_stable_poly = analyzer_stable.stats_each(Path.joinpath(BASE_DATA_DIR,"polygons/each_stable/"))


In [56]:
df_all_stable_polys

,mean,median,mode,std,variance,min,max,skewness,kurtosis,0.5_percentile,99.5_percentile
all_areas,0.519111,0.491943,0.465942,0.410693,0.168669,-27.177246,38.254883,5.91099,404.049652,-0.2146,2.472046


In [57]:
df_each_stable_poly

,mean,median,mode,std,variance,min,max,skewness,kurtosis,0.5_percentile,99.5_percentile
area_id,,,,,,,,,,,
0,0.519111,0.491943,0.465942,0.410693,0.168669,-27.177246,38.254883,5.91099,404.049652,-0.2146,2.472046


<h3><a id="estimate-error"></a>Estimate systematic error</h3>


Systematic error (vertical bias) represents a constant offset between the two DEMs that affects all elevation differences uniformly. This bias can arise from instrument calibration errors, incorrect atmospheric corrections, GNSS/IMU misalignments, or inconsistencies in vertical coordinate reference systems such as mismatched geoid models ([Glennie, 2007](https://doi.org/10.1515/jag.2007.017); [Habib et al., 2009](https://doi.org/10.14358/PERS.75.10.1159)). Geoid errors typically produce shifts of 10–20 cm, while confusion between ellipsoidal and orthometric heights can cause offsets of tens of meters.

**Why use the median?**

We estimate vertical bias as the **median** of elevation differences in stable areas rather than the mean. The median is more robust to outliers(isolated large values from real change, misclassification, or edge effects—ensuring the bias estimate reflects the typical offset rather than being skewed by anomalous values) (Brigham et al.)

**Interpreting the bias:**

- A non-zero median indicates systematic offset that should be removed before variogram analysis
- The bootstrap uncertainty quantifies confidence in the bias estimate
- After bias removal, the distribution should be approximately centered on zero

**Important considerations:**

- If you observe widespread real change (e.g., regional uplift from an earthquake or isostatic rebound), this signal will be absorbed into the bias estimate and must be accounted for separately
- For DSM differencing, vegetation changes (leaf-on vs. leaf-off) can bias the median; consider estimating bias from DTM results instead
- Very large biases (>1 m) may indicate datum errors that should be investigated before proceeding

In [58]:

# Load the stable area raster (this is the masked difference raster)
stable_area_path = Path.joinpath(BASE_DATA_DIR, "polygons/combined_stable.tif")
stable_area = Raster.from_file(stable_area_path)

# Get the median from the stable area
# Read the data directly with rasterio to get valid values

with rasterio.open(stable_area_path) as src:
    data = src.read(1)
    nodata = src.nodata
    # Mask nodata values
    if nodata is not None:
        valid_data = data[data != nodata]
    else:
        valid_data = data[np.isfinite(data)]
    
    diff_stable_median = np.median(valid_data)
    print(f"Median of stable area differences: {diff_stable_median:.4f} m")

# Set up paths and parameters
output_path = Path.joinpath(BASE_DATA_DIR, "polygons/combined_stable_bias_removed.tif")
unit = "m"
dem_resolution = 1.0

# Load raster data using RasterDataHandler
raster_data_handler = RasterDataHandler(stable_area_path, unit, dem_resolution)
raster_data_handler.load_raster()

# Get the data array
vert_diff_array = raster_data_handler.data_array

# Measure of vertical bias (median)
vertical_bias = np.median(vert_diff_array)
print(f"Vertical bias: {vertical_bias:.4f} m")

# Get uncertainty in the median value by bootstrap resampling
stats = StatisticalAnalysis(raster_data_handler)
median_uncertainty = stats.bootstrap_uncertainty_subsample(n_bootstrap=1000, subsample_proportion=0.1)
print(f"Median uncertainty (bootstrap): {median_uncertainty:.4f} m")

# Subtract the vertical bias from the stable area raster and save
raster_data_handler.subtract_value_from_raster(output_path, vertical_bias)
print(f"Saved bias-removed raster to: {output_path}")

# Create new RasterDataHandler for the modified raster
raster_bias_removed = RasterDataHandler(output_path, unit, dem_resolution)
raster_bias_removed.load_raster()

print(f"\nBias-removed stats:")
print(f"  Mean: {np.mean(raster_bias_removed.data_array):.4f} m")
print(f"  Median: {np.median(raster_bias_removed.data_array):.4f} m")
print(f"  Std: {np.std(raster_bias_removed.data_array):.4f} m")

/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Median of stable area differences: nan m
Vertical bias: 0.4919 m
Median uncertainty (bootstrap): 0.0003 m
Saved bias-removed raster to: test_data/paper_examples/az_high_slope/small/polygons/combined_stable_bias_removed.tif

Bias-removed stats:
  Mean: 0.0272 m
  Median: 0.0000 m
  Std: 0.4107 m


In [59]:
fig = stats.plot_data_stats()

<h3><a id="Variography"></a>Variography</h3>


Variography is the process of estimating and modeling the spatial covariance structure of your data. For topographic differencing, we analyze how elevation errors are correlated across space ([Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277); [Oliver & Webster, 2014](https://doi.org/10.1016/j.catena.2013.09.006)).

<h4><a id="semivariogram"></a>The semivariogram</h4>

The **semivariogram** γ(h) measures the average squared difference between values separated by distance h (Matheron, 1965):

$$\gamma(h) = \frac{1}{2N(h)} \sum_{i=1}^{N(h)} [z(x_i) - z(x_i + h)]^2$$

At small distances, nearby points tend to have similar errors (low semivariance). As distance increases, the correlation breaks down and semivariance increases until it reaches a plateau (the sill).

The variogram has several key features:

1. **Nugget (c₀)**: The y-intercept, representing measurement noise and microscale variability below the sampling resolution

2. **Sill (c)**: The plateau value representing total variance. When the variogram reaches the sill, points are no longer spatially correlated.

3. **Range (a)**: The distance at which the sill is reached. Beyond this distance, errors are independent.

<h4><a id="nested_variogram"></a>Nested variograms for multi-scale error</h4> 

Lidar differencing errors operate at multiple scales simultaneously. A **nested variogram** captures this:

$$\gamma(h) = c_0 + \sum_{i=1}^{n} c_i \cdot \text{model}_i(h, a_i)$$

where each component (i) has its own partial sill (cᵢ) and range (aᵢ). For example:
- Component 1 (range ~30 m): Point classification errors
- Component 2 (range ~500 m): Flight line alignment errors
- Component 3 (range ~2 km): Geoid/datum inconsistencies

<h4><a id="model_selection"></a>Model Selection</h4>

We fit nested spherical models and select the best using:
- **AIC (Akaike Information Criterion)**: Balances fit quality vs. model complexity
- **Cross-validation**: Tests predictive performance on held-out data

The spherical model is commonly used because it has a finite range, provides smooth transitions, and handles nesting well ([Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277)).

<h4><a id="sampling"></a>Sampling for computational efficiency</h4>

Computing all pairwise distances for a large raster is computationally prohibitive. We use:
- **Stratified random sampling**: Divide the raster into sub-grids and sample uniformly
- **Multiple realizations**: Repeat sampling ~30 times to estimate confidence bounds ([Ortiz & Deutsch, 2002](https://doi.org/10.1023/A:1014412218427))
- **Numba JIT compilation**: Accelerates the pairwise calculations

In [60]:
#Create variogram analysis instance based on modified raster
V = VariogramAnalysis(raster_bias_removed)

#Calculate a mean variogram with 75 bins from variograms made over 10 runs
V.calculate_mean_variogram_numba(area_side = 250, samples_per_area = 400, max_samples = 1000000000, bin_width = 30, max_n_bins = 3000, n_runs = 30, max_lag_multiplier = 0.5)

/Users/cassandrabrigham/ASU Dropbox/Cassandra Brigham/Mac/Documents/POSTDOC/Code/topographic-differencing-uncertainty/src/topochange/variogram.py:1172: RuntimeWarning: Mean of empty slice
  mean_variogram = np.nanmean(vario_arr, axis=0)
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/cassandrabrigham/ASU Dropbox/Cassandra Brigham/Mac/Documents/POSTDOC/Code/topographic-differencing-uncertainty/src/topochange/variogram.py:1176: RuntimeWarning: Mean of empty slice
  mean_count = np.nanmean(count_arr, axis=0)
/opt/anaconda3/envs/crs_transformation/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [61]:
# Multi-model fitting, compare spherical, exponential, gaussian, matern
print("=" * 70)
print("MULTI-MODEL VARIOGRAM FITTING")
print("=" * 70)

# Fit all candidate models and select best by AIC
best_model = V.fit_best_model_auto(
    model_types=['spherical', 'exponential', 'gaussian', 'matern','hole_effect'],
    max_components=3,
    include_nugget=False,
    criterion='bic',
    compute_cv=True,
    n_bootstrap=500,
    seed=SEED,
)

print(f"\nBest model: {'+'.join(best_model.composite_model.component_names)}")
print(f"AIC: {best_model.aic:.2f}")
print(f"BIC: {best_model.bic:.2f}")
print(f"CV-RMSE: {best_model.cv_rmse:.4f}" if best_model.cv_rmse else "CV-RMSE: N/A")

# Print model parameters
print(f"\n{'─' * 40}")
print("Model Parameters:")
print(f"{'─' * 40}")
cm = best_model.composite_model
print(f"  Nugget: {cm.get_nugget():.6f}")
for i, name in enumerate(cm.component_names):
    comp_params = cm.get_component_params(i)
    param_names = cm.param_names
    # Each component has sill, range (and possibly extra params like nu for matern)
    print(f"\n  Component {i+1}: {name}")
    print(f"    Sill:  {comp_params[0]:.6f}")
    print(f"    Range: {comp_params[1]:.6f}")
    if len(comp_params) > 2:
        for j, val in enumerate(comp_params[2:], start=2):
            print(f"    Param {j}: {val:.6f}")
print(f"\n  Total sill: {cm.get_total_sill():.6f}" if cm.get_total_sill() is not None else "\n  Total sill: N/A (unbounded model)")

MULTI-MODEL VARIOGRAM FITTING

Best model: exponential
AIC: -134.48
BIC: -131.31
CV-RMSE: 0.0228

────────────────────────────────────────
Model Parameters:
────────────────────────────────────────
  Nugget: 0.000000

  Component 1: exponential
    Sill:  0.161529
    Range: 22.068709

  Total sill: 0.161529


In [62]:
fig = V.plot_best_model()

<h3><a id="uncertainty_propagation"></a>Uncertainty propagation</h3>

Once we have a fitted variogram model, we can propagate uncertainty to any area of interest ([Rolstad et al., 2009](https://doi.org/10.3189/002214309789470950)). This is the key step that transforms error characterization into actionable uncertainty bounds.

For a spatially averaged elevation change over an area A, the regional variance σ²_A depends on the spatial covariance structure:

$$\sigma_A^2 = \frac{1}{A^2} \int_A \int_A [\sigma_{\Delta_z}^2 - \gamma(h)] \, dx \, dy$$

**Intuition**: If all points in your area are highly correlated (small area relative to the range), errors move together and uncertainty is high. If points are spread over a large area with mixed positive and negative errors that partially cancel, uncertainty decreases.

A crucial insight: **uncertainty decreases as the area of aggregation increases**, but at different rates depending on the variogram parameters:
- Features smaller than the correlation range: errors are correlated and don't cancel out (uncertainty ≈ √sill)
- Features much larger than the correlation range: errors average toward zero (uncertainty ≈ √(sill / n_effective))

With a nested variogram, each component contributes independently:

$$\sigma_{total}^2 = \sigma_{nugget}^2 + \sigma_{short}^2 + \sigma_{mid}^2 + \sigma_{long}^2$$

For irregularly shaped polygons, we use **Monte Carlo integration**: randomly sample N pairs of points within the polygon, compute covariances using the fitted variogram, and average to approximate the double integral (~10,000–25,000 pairs).

<h4><a id="features_of_interest"></a>Draw features of interest (unstable areas)</h4> 

Use the interactive map below to draw polygons around areas where you expect topographic change (e.g., landslides, construction sites, eroded channels, deposited sediment bars). The uncertainty will be calculated for each polygon.

In [63]:
interactor.map

Map(bottom=3276531.0, center=[36.62094372994652, -112.62951315970199], controls=(ZoomControl(options=['positio…

In [64]:
interactor.unstable_geoms

[<POLYGON ((353944.427 4054137.552, 353969.181 4054178.19, 354008.259 4054219...>]

In [65]:
_, unstable_polys = interactor.export_geodataframes()

In [66]:
# Calculate uncertainty for each feature of interest
# Pass fitted_model so any variogram model type (spherical, matern, hole_effect, etc.) is supported

uncertainties_per_feature = []

for i, poly in enumerate(unstable_polys['geometry']):
    print(f"\nProcessing polygon {i+1}/{len(unstable_polys)}...")

    estimator = RegionalUncertaintyEstimator(
        raster_data_handler=raster_bias_removed,
        variogram_analysis=V,
        area_of_interest=poly,
        fitted_model=V.fitted_model,
    )

    estimator.calc_total_uncertainty(n_pairs=25_000, seed=SEED)
    uncertainties_per_feature.append(estimator)

# Print detailed summary for first polygon
print("\n")
print(uncertainties_per_feature[0].summary())


Processing polygon 1/1...


REGIONAL UNCERTAINTY SUMMARY
Polygon area: 8018.58 m²
Total variance (σ²): 0.161529; min: 0.129223; max: 0.193835

Uncorrelated σ₀: 0.411590
Uncorrelated (polygon mean): 0.004596

POLYGON CORRELATED UNCERTAINTY:
  Total: 0.106723; min: 0.000000; max: 0.221625

POLYGON TOTAL UNCERTAINTY:
  Total: 0.106822; min: 0.004596; max: 0.221673

----------------------------------------------------------------------
RASTER CORRELATED UNCERTAINTY:
  Uncorrelated (raster mean): 0.000348
  Total: 0.000000; min: 0.000000; max: 0.133703

RASTER TOTAL UNCERTAINTY:
  Total: 0.000348; min: 0.000348; max: 0.133704


---

<h2><a id="references"></a>References</h2>

- Albino, F., Smets, B., d'Oreye, N. & Kervyn, F. (2015). High‐resolution TanDEM‐X DEM: An accurate method to estimate lava flow volumes at Nyamulagira Volcano (D.R. Congo). *Journal of Geophysical Research: Solid Earth*, 120, 4189–4207. [https://doi.org/10.1002/2015JB011988](https://doi.org/10.1002/2015JB011988)

- Anderson, S.W. (2019). Uncertainty in quantitative analyses of topographic change: error propagation and the role of thresholding. *Earth Surface Processes and Landforms*, 44, 1015–1033. [https://doi.org/10.1002/esp.4551](https://doi.org/10.1002/esp.4551)

- Besl, P.J. & McKay, N.D. (1992). Method for registration of 3-D shapes. *Sensor Fusion IV: Control Paradigms and Data Structures*, SPIE, 586–606. [https://doi.org/10.1117/12.57955](https://doi.org/10.1117/12.57955)

- Brasington, J. & Smart, R.M.A. (2003). Close range digital photogrammetric analysis of experimental drainage basin evolution. *Earth Surface Processes and Landforms*, 28, 231–247. [https://doi.org/10.1002/esp.480](https://doi.org/10.1002/esp.480)

- Brasington, J., Langham, J. & Rumsby, B. (2003). Methodological sensitivity of morphometric estimates of coarse fluvial sediment transport. *Geomorphology*, 53, 299–316. [https://doi.org/10.1016/S0169-555X(02)00320-3](https://doi.org/10.1016/S0169-555X(02)00320-3)

- Dehecq, A., Gardner, A.S., Alexandrov, O., McMichael, S., Hugonnet, R., Shean, D. & Marty, M. (2020). Automated Processing of Declassified KH-9 Hexagon Satellite Images for Global Elevation Change Analysis Since the 1970s. *Frontiers in Earth Science*, 8, 566802. [https://doi.org/10.3389/feart.2020.566802](https://doi.org/10.3389/feart.2020.566802)

- Glennie, C.L., Hinojosa‐Corona, A., Nissen, E., Kusari, A., Oskin, M.E., Arrowsmith, J.R. & Borsa, A. (2014). Optimization of legacy lidar data sets for measuring near‐field earthquake displacements. *Geophysical Research Letters*, 41, 3494–3501. [https://doi.org/10.1002/2014GL059919](https://doi.org/10.1002/2014GL059919)

- Glennie, C. (2007). Rigorous 3D error analysis of kinematic scanning LIDAR systems. *Journal of Applied Geodesy*, 1, 147–157. [https://doi.org/10.1515/jag.2007.017](https://doi.org/10.1515/jag.2007.017)

- Heritage, G.L., Milan, D.J., Large, A.R.G. & Fuller, I.C. (2009). Influence of survey strategy and interpolation model on DEM quality. *Geomorphology*, 112, 334–344. [https://doi.org/10.1016/j.geomorph.2009.06.024](https://doi.org/10.1016/j.geomorph.2009.06.024)

- Hugonnet, R., Brun, F., Berthier, E., Dehecq, A., Mannerfelt, E.S., Eckert, N. & Farinotti, D. (2022). Uncertainty analysis of digital elevation models by spatial inference from stable terrain. *IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing*, 15, 6456–6472. [https://doi.org/10.1109/JSTARS.2022.3188922](https://doi.org/10.1109/JSTARS.2022.3188922)

- Izumida, A., Uchiyama, S. & Sugai, T. (2017). Application of UAV-SfM photogrammetry and aerial lidar to a disastrous flood: repeated topographic measurement of a midstream 2 river during a recovery. *Natural Hazards and Earth System Sciences*, 17, 1505–1519. [https://doi.org/10.5194/nhess-17-1505-2017](https://doi.org/10.5194/nhess-17-1505-2017)

- Journel, A.G. & Huijbregts, C.J. (1978). *Mining Geostatistics*. Academic Press.

- Lane, S.N., Westaway, R.M. & Hicks, D.M. (2003). Estimation of erosion and deposition volumes in a large, gravel‐bed, braided river using synoptic remote sensing. *Earth Surface Processes and Landforms*, 28, 249–271. [https://doi.org/10.1002/esp.483](https://doi.org/10.1002/esp.483)

- Langridge, R.M., Ries, W.F., Farrier, T., Barth, N.C., Khajavi, N. & De Pascale, G.P. (2014). Developing sub 5-m lidar DEMs for forested sections of the Alpine and Hope faults, South Island, New Zealand. *Geomorphology*, 226, 226–240. [https://doi.org/10.1016/j.geomorph.2014.08.007](https://doi.org/10.1016/j.geomorph.2014.08.007)

- Lucieer, A., de Jong, S.M. & Turner, D. (2014). Mapping landslide displacements using Structure from Motion (SfM) and image correlation of multi-temporal UAV photography. *Progress in Physical Geography*, 38, 97–116. [https://doi.org/10.1177/0309133313515293](https://doi.org/10.1177/0309133313515293)

- Matheron, G. (1965). *Les variables régionalisées et leur estimation*. Masson, Paris.

- Oliver, M.A. & Webster, R. (2014). A tutorial guide to geostatistics: Computing and modelling variograms and kriging. *Catena*, 113, 56–69. [https://doi.org/10.1016/j.catena.2013.09.006](https://doi.org/10.1016/j.catena.2013.09.006)

- Oliver, M.A. & Webster, R. (2015). *Basic Steps in Geostatistics: The Variogram and Kriging*. Springer. [https://doi.org/10.1007/978-3-319-15865-5](https://doi.org/10.1007/978-3-319-15865-5)

- Ortiz, J.M. & Deutsch, C.V. (2002). Calculation of uncertainty in the variogram. *Mathematical Geology*, 34(2), 169–183. [https://doi.org/10.1023/A:1014412218427](https://doi.org/10.1023/A:1014412218427)

- Passalacqua, P., Belmont, P., Staley, D.M. et al. (2015). Analyzing high resolution topography for advancing the understanding of mass and energy transfer through landscapes: A review. *Earth-Science Reviews*, 148, 174–193. [https://doi.org/10.1016/j.earscirev.2015.05.012](https://doi.org/10.1016/j.earscirev.2015.05.012)

- Rolstad, C., Haug, T. & Denby, B. (2009). Spatially integrated geodetic glacier mass balance and its uncertainty based on geostatistical analysis: application to the western Svartisen ice cap, Norway. *Journal of Glaciology*, 55(192), 666–680. [https://doi.org/10.3189/002214309789470950](https://doi.org/10.3189/002214309789470950)

- Schaffrath, K.R., Belmont, P. & Wheaton, J.M. (2015). Landscape-scale geomorphic change detection: Quantifying spatially variable uncertainty and circumventing legacy data issues. *Geomorphology*, 250, 334–348. [https://doi.org/10.1016/j.geomorph.2015.09.020](https://doi.org/10.1016/j.geomorph.2015.09.020)

- Scott, C., Arrowsmith, J.R., Nissen, E., Lajoie, L., Maruyama, T. & Chiba, T. (2018). The M7 2016 Kumamoto, Japan, Earthquake: 3-D Deformation Along the Fault and Within the Damage Zone Constrained From Differential Lidar Topography. *Journal of Geophysical Research: Solid Earth*, 123, 6138–6155. [https://doi.org/10.1029/2018JB015581](https://doi.org/10.1029/2018JB015581)

- Scott, C., Phan, M., Nandigam, V., Crosby, C. & Arrowsmith, J.R. (2021). Measuring change at Earth's surface: On-demand vertical and three-dimensional topographic differencing implemented in OpenTopography. *Geosphere*, 17, 1318–1332. [https://doi.org/10.1130/GES02259.1](https://doi.org/10.1130/GES02259.1)

- Webster, R. & Oliver, M.A. (2007). *Geostatistics for Environmental Scientists*, 2nd ed. John Wiley & Sons. [https://doi.org/10.1002/9780470517277](https://doi.org/10.1002/9780470517277)

- Wheaton, J.M., Brasington, J., Darby, S.E. & Sear, D.A. (2010). Accounting for uncertainty in DEMs from repeat topographic surveys: improved sediment budgets. *Earth Surface Processes and Landforms*, 35, 136–156. [https://doi.org/10.1002/esp.1886](https://doi.org/10.1002/esp.1886)

